In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("DIM_CUSTOMER") # default 4
spark.getActiveSession()

In [3]:
cc.set_connectionProfile("VeloDB")

df_operational_users = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()


df_operational_subscription_type = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_operational_subscriptions = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()


In [4]:
# df_operational_users.createOrReplaceTempView("dimUsers")
# df_operational_subscriptions.createOrReplaceTempView("dimSubscriptions")
# df_operational_subscription_type.createOrReplaceTempView("dimSubscriptionTypes")


In [5]:
#TEMP VIEW
df_operational_users.createOrReplaceTempView("dimCustomer")
df_operational_subscription_type.createOrReplaceTempView("subscriptionTypes")
df_operational_subscriptions.createOrReplaceTempView("subscriptions")

# TRANSFORM: First, create a temp view for the most recent subscription per user
spark.sql("""

SELECT userId, validfrom, description
FROM (
SELECT 
        s.userId, 
        s.validfrom, 
        st.description,
        ROW_NUMBER() OVER (PARTITION BY s.userId ORDER BY s.validfrom DESC) AS rank
    FROM subscriptions AS s
    LEFT JOIN subscriptionTypes AS st ON s.subscriptiontypeid = st.subscriptiontypeid
)
WHERE rank = 1
""").createOrReplaceTempView("mostRecentSubscription")

# Final dimension: Join the customer and most recent subscription details
df_dim_customer = spark.sql("""
SELECT 
    uuid() AS customerSK, 
    c.userId, 
    c.name, 
    c.email, 
    c.street, 
    c.number, 
    c.zipcode, 
    c.city, 
    c.country_code, 
    m.validfrom, 
    m.description, 
    to_timestamp('1999-01-01','yyyy-MM-dd') AS scd_start, 
    to_timestamp('2100-12-12','yyyy-MM-dd') AS scd_end, 
    md5(concat(c.name, c.email, c.street, c.number, c.zipcode, c.city, c.country_code)) AS md5, 
    True AS current
FROM dimCustomer AS c
LEFT JOIN mostRecentSubscription AS m ON c.userId = m.userId
""")

# Create temp view for the final dimension
df_dim_customer.createOrReplaceTempView("dimCustomer")

# Display the final dimension
spark.sql("SELECT * FROM dimCustomer").show()

+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|          customerSK|userId|                name|               email|              street|  number|zipcode|                city|country_code| validfrom|description|          scd_start|            scd_end|                 md5|current|
+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|710a7a56-4f27-4d3...|     2|   van der Zee Julia|Julia.van.der.Zee...|          Europalaan|     43 |   2610| Wilrijk (Antwerpen)|          BE|2021-10-05|      MAAND|1999-01-01 00:00:00|2100-12-12 00:00:00|01301120bda974125...|   true|
|5849bcb8-2a2e-481...|     4|      Willems Angela|Angela

In [6]:
# df_dim_users = spark.sql("select uuid() as customerSK, c.userId, c.name, c.email, c.street, c.number, c.zipcode, c.city, c.country_code, st.description, s.validfrom, to_timestamp('1999-01-01','yyyy-MM-dd') as scd_start, to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end, md5(concat(c.name, c.email, c.street, c.number, c.zipcode, c.city, c.country_code)) as md5, True as current  from dimUsersRep as c LEFT JOIN dimSubscriptions as s on c.userId = s.userId LEFT JOIN dimSubscriptionTypes as st ON s.subscriptiontypeid = st.subscriptiontypeid")


In [7]:
df_dim_customer.createOrReplaceTempView("dimCustomer")
spark.sql("select * from dimCustomer").show()

+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|          customerSK|userId|                name|               email|              street|  number|zipcode|                city|country_code| validfrom|description|          scd_start|            scd_end|                 md5|current|
+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+----------+-----------+-------------------+-------------------+--------------------+-------+
|710a7a56-4f27-4d3...|     2|   van der Zee Julia|Julia.van.der.Zee...|          Europalaan|     43 |   2610| Wilrijk (Antwerpen)|          BE|2021-10-05|      MAAND|1999-01-01 00:00:00|2100-12-12 00:00:00|01301120bda974125...|   true|
|5849bcb8-2a2e-481...|     4|      Willems Angela|Angela

In [10]:
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("dimCustomer")
